# Bài tập 5 — Mục 4: Tối ưu hóa bằng Materialized Views
**Materialized Views — Query Rewrite — Refresh — Incremental View Maintenance**

**Nguồn tài liệu:** R. Elmasri & S. B. Navathe, *Fundamentals of Database Systems*, 7th ed.

- Chương 7, mục 7.3.1–7.3.3, View thường và định nghĩa `DEPT_INFO` (V2): **PDF 259–261** (trang sách 228–230).
- Chương 19, mục 19.2.4, Materialized Views: **PDF 738–741** (trang sách 707–710).
- Dữ liệu COMPANY, Figure 5.5 và 5.6: **PDF 192–193** (trang sách 161–162).


## 1. Tạo COMPANY bằng dữ liệu trong Figure 5.6
Chỉ cần `DEPARTMENT` và `EMPLOYEE` để tái hiện **V2: DEPT_INFO** của sách.

In [ ]:
import sqlite3

# CSDL trong RAM: mỗi lần Run all sẽ tạo lại từ đầu; không cần file .db ngoài.
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")

def rows(sql, parameters=()):
    return [tuple(row) for row in conn.execute(sql, parameters).fetchall()]

def show(title, sql, parameters=()):
    cur = conn.execute(sql, parameters)
    values = [tuple(v) for v in cur.fetchall()]
    headers = [col[0] for col in cur.description]
    print("\n" + title)
    if not values:
        print("(0 dòng)")
        return values
    widths = [max(len(str(h)), *(len(str(v[i])) for v in values)) for i,h in enumerate(headers)]
    print(" | ".join(str(h).ljust(widths[i]) for i,h in enumerate(headers)))
    print("-+-".join("-"*w for w in widths))
    for value in values:
        print(" | ".join(str(v).ljust(widths[i]) for i,v in enumerate(value)))
    print(f"Tổng: {len(values)} dòng")
    return values

def show_plan(title, sql):
    print("\n" + title)
    for row in conn.execute("EXPLAIN QUERY PLAN " + sql):
        print("  ", row[3])

In [ ]:
conn.executescript("""
DROP VIEW IF EXISTS WORKS_ON1;
DROP TABLE IF EXISTS WORKS_ON;
DROP TABLE IF EXISTS PROJECT;
DROP TABLE IF EXISTS EMPLOYEE;
DROP TABLE IF EXISTS DEPARTMENT;

CREATE TABLE DEPARTMENT (
    Dname TEXT NOT NULL,
    Dnumber INTEGER PRIMARY KEY
);
CREATE TABLE EMPLOYEE (
    Fname TEXT NOT NULL,
    Lname TEXT NOT NULL,
    Ssn TEXT PRIMARY KEY,
    Salary INTEGER NOT NULL,
    Dno INTEGER NOT NULL,
    FOREIGN KEY (Dno) REFERENCES DEPARTMENT(Dnumber)
);

INSERT INTO DEPARTMENT VALUES\n  ('Research', 5),\n  ('Administration', 4),\n  ('Headquarters', 1);
INSERT INTO EMPLOYEE VALUES\n  ('John', 'Smith', '123456789', 30000, 5),\n  ('Franklin', 'Wong', '333445555', 40000, 5),\n  ('Alicia', 'Zelaya', '999887777', 25000, 4),\n  ('Jennifer', 'Wallace', '987654321', 43000, 4),\n  ('Ramesh', 'Narayan', '666884444', 38000, 5),\n  ('Joyce', 'English', '453453453', 25000, 5),\n  ('Ahmad', 'Jabbar', '987987987', 25000, 4),\n  ('James', 'Borg', '888665555', 55000, 1);
""")
print("Đã tạo COMPANY từ Figure 5.6 của sách.")
print("DEPARTMENT:", conn.execute("SELECT COUNT(*) FROM DEPARTMENT").fetchone()[0], "dòng")
print("EMPLOYEE:", conn.execute("SELECT COUNT(*) FROM EMPLOYEE").fetchone()[0], "dòng")

Đã tạo COMPANY từ Figure 5.6 của sách.
DEPARTMENT: 3 dòng
EMPLOYEE: 8 dòng


In [ ]:
show("Dữ liệu phòng ban", "SELECT * FROM DEPARTMENT ORDER BY Dnumber")
show("Nhân viên và mức lương trong Figure 5.6", "SELECT Fname,Lname,Salary,Dno FROM EMPLOYEE ORDER BY Ssn")


Dữ liệu phòng ban
Dname          | Dnumber
---------------+--------
Headquarters   | 1      
Administration | 4      
Research       | 5      
Tổng: 3 dòng

Nhân viên và mức lương trong Figure 5.6
Fname    | Lname   | Salary | Dno
---------+---------+--------+----
John     | Smith   | 30000  | 5  
Franklin | Wong    | 40000  | 5  
Joyce    | English | 25000  | 5  
Ramesh   | Narayan | 38000  | 5  
James    | Borg    | 55000  | 1  
Jennifer | Wallace | 43000  | 4  
Ahmad    | Jabbar  | 25000  | 4  
Alicia   | Zelaya  | 25000  | 4  
Tổng: 8 dòng


[('John', 'Smith', 30000, 5),
 ('Franklin', 'Wong', 40000, 5),
 ('Joyce', 'English', 25000, 5),
 ('Ramesh', 'Narayan', 38000, 5),
 ('James', 'Borg', 55000, 1),
 ('Jennifer', 'Wallace', 43000, 4),
 ('Ahmad', 'Jabbar', 25000, 4),
 ('Alicia', 'Zelaya', 25000, 4)]

## 2. View thường so với kết quả đã vật lý hóa
**Mục 7.3.2:** sách đưa ra View `DEPT_INFO(Dept_name, No_of_emps, Total_sal)`, thống kê **số nhân viên** và **tổng lương** của từng phòng ban bằng `JOIN` và `GROUP BY`.

**Mục 19.2.4:** nếu kết quả này cần sử dụng thường xuyên, có thể lưu kết quả để đọc lại, tránh tính JOIN và phép tổng hợp từ đầu trên mỗi lần dùng.

Notebook tạo:
1. `DEPT_INFO` = **View thông thường** theo V2 trong sách.
2. `DEPT_INFO_MAT` = **bảng chứa bản sao kết quả tính sẵn** (mô phỏng một Materialized View).

In [ ]:
conn.executescript("""
CREATE VIEW DEPT_INFO(Dept_name, No_of_emps, Total_sal) AS
SELECT D.Dname, COUNT(*), SUM(E.Salary)
FROM DEPARTMENT D
JOIN EMPLOYEE E ON D.Dnumber = E.Dno
GROUP BY D.Dname;

-- SQLite: dùng bảng vật lý để minh họa materialization.
CREATE TABLE DEPT_INFO_MAT AS
SELECT Dept_name, No_of_emps, Total_sal FROM DEPT_INFO;
""")
q_view = "SELECT Dept_name, No_of_emps, Total_sal FROM DEPT_INFO ORDER BY Dept_name"
q_mat  = "SELECT Dept_name, No_of_emps, Total_sal FROM DEPT_INFO_MAT ORDER BY Dept_name"
realtime = show("VIEW: tính thông qua định nghĩa SELECT",q_view)
stored = show("MATERIALIZED (mô phỏng): đọc kết quả đã lưu",q_mat)
assert realtime == stored
print("✓ Kết quả ban đầu khớp nhau.")
show_plan("Execution plan SQLite — View thông thường",q_view)
show_plan("Execution plan SQLite — bảng kết quả đã lưu",q_mat)


VIEW: tính thông qua định nghĩa SELECT
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 3          | 93000    
Headquarters   | 1          | 55000    
Research       | 4          | 133000   
Tổng: 3 dòng

MATERIALIZED (mô phỏng): đọc kết quả đã lưu
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 3          | 93000    
Headquarters   | 1          | 55000    
Research       | 4          | 133000   
Tổng: 3 dòng
✓ Kết quả ban đầu khớp nhau.

Execution plan SQLite — View thông thường
   CO-ROUTINE DEPT_INFO
   SCAN E
   SEARCH D USING INTEGER PRIMARY KEY (rowid=?)
   USE TEMP B-TREE FOR GROUP BY
   SCAN DEPT_INFO
   USE TEMP B-TREE FOR ORDER BY

Execution plan SQLite — bảng kết quả đã lưu
   SCAN DEPT_INFO_MAT
   USE TEMP B-TREE FOR ORDER BY


## 3. Vì sao phải REFRESH?
**Theo mục 19.2.4, trang sách 707:** khi dữ liệu bảng gốc thay đổi, View đã lưu có thể **cũ**. Có ba chiến lược trong sách:

- **Immediate update:** cập nhật ngay khi bảng gốc thay đổi.
- **Lazy update:** chỉ tính lại khi cần.
- **Periodic / deferred update:** cập nhật sau, có thể theo lịch.

Để nhìn thấy tình huống dữ liệu cũ, chương trình **thay đổi giả định** mức lương John Smith từ `30000` thành `40000` (cả hai đều là giá trị xuất hiện trong Figure 5.6). Đây là **tình huống cập nhật thử**, không phải dữ liệu mới mà sách khẳng định.

In [ ]:
with conn:
    conn.execute("UPDATE EMPLOYEE SET Salary = 40000 WHERE Ssn = '123456789'")
current = show("Sau khi sửa bảng gốc — VIEW phản ánh lương mới",q_view)
old = show("Bảng vật lý hóa CHƯA REFRESH — dữ liệu cũ",q_mat)
assert current != old
print("✓ Đã chứng minh kết quả lưu sẵn có thể cũ khi chưa được cập nhật.")


Sau khi sửa bảng gốc — VIEW phản ánh lương mới
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 3          | 93000    
Headquarters   | 1          | 55000    
Research       | 4          | 143000   
Tổng: 3 dòng

Bảng vật lý hóa CHƯA REFRESH — dữ liệu cũ
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 3          | 93000    
Headquarters   | 1          | 55000    
Research       | 4          | 133000   
Tổng: 3 dòng
✓ Đã chứng minh kết quả lưu sẵn có thể cũ khi chưa được cập nhật.


## 4. Full Refresh: tính lại toàn bộ kết quả
Một phương án đơn giản là tạo lại kết quả View từ các bảng gốc. Sách lưu ý cách **tính lại toàn bộ** có thể tốn kém nếu thay đổi xảy ra thường xuyên; vì thế phần tiếp theo sẽ minh họa cập nhật gia tăng.

In [ ]:
with conn:
    conn.execute("DELETE FROM DEPT_INFO_MAT")
    conn.execute("""
        INSERT INTO DEPT_INFO_MAT(Dept_name, No_of_emps, Total_sal)
        SELECT Dept_name, No_of_emps, Total_sal FROM DEPT_INFO
    """)
full_refreshed = show("Sau Full Refresh", q_mat)
assert full_refreshed == rows(q_view)
print("✓ Bảng vật lý hóa lại khớp với View trên dữ liệu gốc.")


Sau Full Refresh
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 3          | 93000    
Headquarters   | 1          | 55000    
Research       | 4          | 143000   
Tổng: 3 dòng
✓ Bảng vật lý hóa lại khớp với View trên dữ liệu gốc.


## 5. Incremental View Maintenance: chỉ xử lý phần thay đổi
**Theo mục 19.2.4, trang sách 708–709:** đối với `SUM` và `COUNT` theo nhóm, có thể điều chỉnh **tổng** và **số lượng** khi một bộ dữ liệu được thêm, xóa hoặc sửa, không cần dựng lại toàn bộ View.

Thử nghiệm 1: thay đổi lương Joyce English (`25000` → `30000`), cùng hai giá trị có sẵn trong Figure 5.6. Vì số nhân viên không đổi, chỉ cộng mức chênh lệch lương vào `Total_sal`. Việc cập nhật bảng gốc và bảng lưu kết quả thực hiện **cùng một giao dịch** để tránh dữ liệu thiếu nhất quán.

In [ ]:
ssn = '453453453'  # Joyce English trong Figure 5.6
old_salary, dno = tuple(conn.execute(
    "SELECT Salary, Dno FROM EMPLOYEE WHERE Ssn=?",(ssn,)
).fetchone())
new_salary = 30000

delta = new_salary - old_salary
with conn:
    conn.execute("UPDATE EMPLOYEE SET Salary=? WHERE Ssn=?", (new_salary,ssn))
    conn.execute("""
        UPDATE DEPT_INFO_MAT
        SET Total_sal = Total_sal + ?
        WHERE Dept_name = (SELECT Dname FROM DEPARTMENT WHERE Dnumber = ?)
    """, (delta,dno))
show("Sau cập nhật gia tăng lương (+delta)", q_mat)
assert rows(q_view) == rows(q_mat)
print(f"✓ Delta = {new_salary} - {old_salary} = {delta}. Không cần tính lại toàn bộ bảng kết quả.")


Sau cập nhật gia tăng lương (+delta)
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 3          | 93000    
Headquarters   | 1          | 55000    
Research       | 4          | 148000   
Tổng: 3 dòng
✓ Delta = 30000 - 25000 = 5000. Không cần tính lại toàn bộ bảng kết quả.


Thử nghiệm 2: chuyển một nhân viên **đã có trong sách** từ phòng `Research` (`Dno=5`) sang phòng `Administration` (`Dno=4`). Cách diễn giải cập nhật như **xóa ảnh hưởng khỏi nhóm cũ, rồi thêm ảnh hưởng vào nhóm mới** phù hợp với mô tả cập nhật gia tăng trong mục 19.2.4.

Không thêm người mới hay mức lương tự tạo. Cập nhật cả `No_of_emps` và `Total_sal` cho hai nhóm.

In [ ]:
ssn = '666884444'  # Ramesh Narayan, Figure 5.6
old_dno, salary = tuple(conn.execute(
    "SELECT Dno, Salary FROM EMPLOYEE WHERE Ssn = ?",(ssn,)
).fetchone())
new_dno = 4
with conn:
    conn.execute("UPDATE EMPLOYEE SET Dno=? WHERE Ssn=?",(new_dno,ssn))
    conn.execute("""
        UPDATE DEPT_INFO_MAT
        SET No_of_emps = No_of_emps - 1, Total_sal = Total_sal - ?
        WHERE Dept_name = (SELECT Dname FROM DEPARTMENT WHERE Dnumber = ?)
    """,(salary,old_dno))
    conn.execute("""
        UPDATE DEPT_INFO_MAT
        SET No_of_emps = No_of_emps + 1, Total_sal = Total_sal + ?
        WHERE Dept_name = (SELECT Dname FROM DEPARTMENT WHERE Dnumber = ?)
    """,(salary,new_dno))
show("Sau khi chuyển phòng — cập nhật 2 nhóm có liên quan",q_mat)
assert rows(q_view)==rows(q_mat)
print("✓ Hai nhóm cập nhật đúng số lượng và tổng lương; nhóm còn lại không phải tính lại.")


Sau khi chuyển phòng — cập nhật 2 nhóm có liên quan
Dept_name      | No_of_emps | Total_sal
---------------+------------+----------
Administration | 4          | 131000   
Headquarters   | 1          | 55000    
Research       | 3          | 110000   
Tổng: 3 dòng
✓ Hai nhóm cập nhật đúng số lượng và tổng lương; nhóm còn lại không phải tính lại.


## 6. Kiểm tra cuối và kết luận
Các phép so sánh `assert` ở từng bước chứng minh dữ liệu sau bảo trì khớp với việc tính từ bảng gốc. Đây là minh họa **đúng nguyên lý sách**, không phải phép đo hiệu năng trên dữ liệu lớn.

- **View thường:** truy vấn dựa trên định nghĩa `SELECT`, phản ánh dữ liệu bảng gốc.
- **Materialized View (mô phỏng bằng bảng):** đọc kết quả đã lưu, có thể tránh tính toán lại các phép JOIN và tổng hợp.
- **Refresh:** cần thiết khi bảng gốc thay đổi; `Immediate`, `Lazy`, `Periodic` là ba chiến lược sách nêu.
- **Incremental View Maintenance:** cập nhật phần chênh lệch (`SUM`, `COUNT`) thay vì tính lại toàn bộ; lợi ích phải cân nhắc cùng chi phí bảo trì.

**Nguồn:** Elmasri & Navathe, *Fundamentals of Database Systems*, 7th ed., mục 7.3 và 19.2.4.